In [1]:
import time

import bs4
import pandas as pd
import requests
import numpy as np
import nltk
import matplotlib.pyplot as plt
import seaborn as sns
import gnews
import yfinance as yf
from gnews import GNews
from lxml.html.diff import href_token
import bs4 as bs
import json
import csv

In [2]:
news=GNews(start_date=(2019,11,4), end_date=(2024,11,2),max_results=1000)
ns=news.get_news("Microsoft")

C:\Users\Asus\anaconda3\Lib\site-packages\gnews\gnews.py:218: UserWarning: Only searches using get_news support date ranges. Start and end dates will be ignored.
  return self._get_news_more_than_100(key)


In [1]:
news_df=pd.DataFrame(data=ns)
news_df.head()

NameError: name 'pd' is not defined

In [4]:
news_df.shape

(1000, 5)

In [40]:
temp=news_df
temp["full_text"] = (temp["title"].fillna("") + "" +temp["description"].fillna(""))

In [41]:
temp.head()

,title,description,published date,url,publisher,full_text
0,Microsoft Bans Term ‘Microslop’ From Official ...,Microsoft Bans Term ‘Microslop’ From Official ...,"Mon, 02 Mar 2026 19:40:56 GMT",https://news.google.com/rss/articles/CBMikwFBV...,"{'href': 'https://gizmodo.com', 'title': 'Gizm...",Microsoft Bans Term ‘Microslop’ From Official ...
1,"Microsoft says stop calling it Microslop, or y...","Microsoft says stop calling it Microslop, or y...","Mon, 02 Mar 2026 15:35:00 GMT",https://news.google.com/rss/articles/CBMiowFBV...,"{'href': 'https://www.pcworld.com', 'title': '...","Microsoft says stop calling it Microslop, or y..."
2,Why the Microsoft 365 Copilot bug matters for ...,Why the Microsoft 365 Copilot bug matters for ...,"Mon, 02 Mar 2026 15:37:32 GMT",https://news.google.com/rss/articles/CBMihwFBV...,"{'href': 'https://www.foxnews.com', 'title': '...",Why the Microsoft 365 Copilot bug matters for ...
3,Microsoft ends data center non-disclosure agre...,Microsoft ends data center non-disclosure agre...,"Tue, 03 Mar 2026 19:22:00 GMT",https://news.google.com/rss/articles/CBMi6wFBV...,"{'href': 'https://www.detroitnews.com', 'title...",Microsoft ends data center non-disclosure agre...
4,Microsoft’s big developer conference returns t...,Microsoft’s big developer conference returns t...,"Tue, 03 Mar 2026 17:00:00 GMT",https://news.google.com/rss/articles/CBMiggFBV...,"{'href': 'https://www.theverge.com', 'title': ...",Microsoft’s big developer conference returns t...


In [48]:
temp.iloc[2,1]

'Why the Microsoft 365 Copilot bug matters for data security  Fox News'

In [8]:
news_df.to_csv(r"C:\Users\Asus\Documents\Skills\Python\Sentiment Analysis\MSFT_news.csv")

In [2]:
subreddit=["https://www.reddit.com/r/wallstreetbets/","https://www.reddit.com/r/stocks/"]

In [3]:
def scrape_reddit(subreddit:str,total_posts:int):

    all_data=[]         #main list
    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36"}     #framework

    for index,url in enumerate(subreddit):
        sr_name=url.split("/")[-2]      #names of subreddit
        after=None
        posts=[]
        print(sr_name)

        while len(all_data) < total_posts:

            params = {"limit": 100}
            if after:
                params["after"] = after

            try:
                response=requests.get(f"https://www.reddit.com/r/{sr_name}/.json",headers=headers,timeout=15,params=params)
                res_json=response.json()
                print(res_json)
                data = res_json["data"]["children"]
                response.raise_for_status()


                for post in data:
                    p = post["data"]
                    posts.append({"subreddit": sr_name,"title": p["title"],"url": "https://reddit.com" + p["permalink"],"score": p["score"],"num_comments": p["num_comments"],"created": p["created_utc"]})

                after = res_json["data"]["after"]

                if after is None:
                    break
                time.sleep(1)

                all_data.append(posts)


            except Exception as e:
                print(e)

    return all_data

def save_scraped_data(data, filename_json, filename_csv):
    if not data:
        print(f'No data')
        return


    try:
        with open(filename_json, "w", encoding="utf-8") as file:
            json.dump(data, file, indent=2, ensure_ascii=True)
        print(f'Topics saved: {filename_json}')
    except Exception as e:
        print("ERROR:",e)



    try:
        with open(filename_csv, 'w', newline='', encoding='utf-8') as file:
            writer = csv.writer(file)
            writer.writerow(['Subreddit', 'Type', 'Title', 'URL'])

            for subreddit_data in data:
                subreddit = subreddit_data['name']

                for topic in subreddit_data.get('topics', []):
                    writer.writerow([subreddit, topic['type'], topic['title'], ''])

                for discussion in subreddit_data.get('discussions', []):
                    writer.writerow([subreddit, discussion['type'], discussion['title'], discussion['url']])

            print(f'All topics saved to files!')
    except Exception as e:
        print("Error:",e)

In [4]:
a=scrape_reddit(subreddit,5)
a

[[{'subreddit': 'wallstreetbets',
   'title': 'Weekly Earnings Threads 3/2 - 3/6',
   'url': 'https://reddit.com/r/wallstreetbets/comments/1rgbpm8/weekly_earnings_threads_32_36/',
   'score': 192,
   'num_comments': 1236,
   'created': 1772210511.0},
  {'subreddit': 'wallstreetbets',
   'title': 'What Are Your Moves Tomorrow, March 06, 2026',
   'url': 'https://reddit.com/r/wallstreetbets/comments/1rltgpe/what_are_your_moves_tomorrow_march_06_2026/',
   'score': 169,
   'num_comments': 5158,
   'created': 1772744230.0},
  {'subreddit': 'wallstreetbets',
   'title': 'US grants waiver to allow India to buy Russian oil amid Iran war',
   'url': 'https://reddit.com/r/wallstreetbets/comments/1rm15iz/us_grants_waiver_to_allow_india_to_buy_russian/',
   'score': 1864,
   'num_comments': 160,
   'created': 1772762834.0},
  {'subreddit': 'wallstreetbets',
   'title': 'The market lately',
   'url': 'https://reddit.com/r/wallstreetbets/comments/1rlz6d2/the_market_lately/',
   'score': 1793,
   'n

In [6]:
save_scraped_data(a,"smth.json","smth.csv")

Topics saved: smth.json
Error: list indices must be integers or slices, not str


In [7]:
f=pd.read_json("smth.json")
f

,0,1,2,3,4,5,6,7,8,9,...,490,491,492,493,494,495,496,497,498,499
0,"{'subreddit': 'wallstreetbets', 'title': 'Week...","{'subreddit': 'wallstreetbets', 'title': 'What...","{'subreddit': 'wallstreetbets', 'title': 'US g...","{'subreddit': 'wallstreetbets', 'title': 'The ...","{'subreddit': 'wallstreetbets', 'title': 'Orac...","{'subreddit': 'wallstreetbets', 'title': 'US w...","{'subreddit': 'wallstreetbets', 'title': 'Why ...","{'subreddit': 'wallstreetbets', 'title': 'US D...","{'subreddit': 'wallstreetbets', 'title': 'Man,...","{'subreddit': 'wallstreetbets', 'title': 'FUCK...",...,"{'subreddit': 'wallstreetbets', 'title': 'Mode...","{'subreddit': 'wallstreetbets', 'title': 'Am I...","{'subreddit': 'wallstreetbets', 'title': '32k ...","{'subreddit': 'wallstreetbets', 'title': 'some...","{'subreddit': 'wallstreetbets', 'title': 'Dail...","{'subreddit': 'wallstreetbets', 'title': 'Casi...","{'subreddit': 'wallstreetbets', 'title': 'Rega...","{'subreddit': 'wallstreetbets', 'title': 'Shop...","{'subreddit': 'wallstreetbets', 'title': 'I th...","{'subreddit': 'wallstreetbets', 'title': 'CVNA..."
1,"{'subreddit': 'wallstreetbets', 'title': 'Week...","{'subreddit': 'wallstreetbets', 'title': 'What...","{'subreddit': 'wallstreetbets', 'title': 'US g...","{'subreddit': 'wallstreetbets', 'title': 'The ...","{'subreddit': 'wallstreetbets', 'title': 'Orac...","{'subreddit': 'wallstreetbets', 'title': 'US w...","{'subreddit': 'wallstreetbets', 'title': 'Why ...","{'subreddit': 'wallstreetbets', 'title': 'US D...","{'subreddit': 'wallstreetbets', 'title': 'Man,...","{'subreddit': 'wallstreetbets', 'title': 'FUCK...",...,"{'subreddit': 'wallstreetbets', 'title': 'Mode...","{'subreddit': 'wallstreetbets', 'title': 'Am I...","{'subreddit': 'wallstreetbets', 'title': '32k ...","{'subreddit': 'wallstreetbets', 'title': 'some...","{'subreddit': 'wallstreetbets', 'title': 'Dail...","{'subreddit': 'wallstreetbets', 'title': 'Casi...","{'subreddit': 'wallstreetbets', 'title': 'Rega...","{'subreddit': 'wallstreetbets', 'title': 'Shop...","{'subreddit': 'wallstreetbets', 'title': 'I th...","{'subreddit': 'wallstreetbets', 'title': 'CVNA..."
2,"{'subreddit': 'wallstreetbets', 'title': 'Week...","{'subreddit': 'wallstreetbets', 'title': 'What...","{'subreddit': 'wallstreetbets', 'title': 'US g...","{'subreddit': 'wallstreetbets', 'title': 'The ...","{'subreddit': 'wallstreetbets', 'title': 'Orac...","{'subreddit': 'wallstreetbets', 'title': 'US w...","{'subreddit': 'wallstreetbets', 'title': 'Why ...","{'subreddit': 'wallstreetbets', 'title': 'US D...","{'subreddit': 'wallstreetbets', 'title': 'Man,...","{'subreddit': 'wallstreetbets', 'title': 'FUCK...",...,"{'subreddit': 'wallstreetbets', 'title': 'Mode...","{'subreddit': 'wallstreetbets', 'title': 'Am I...","{'subreddit': 'wallstreetbets', 'title': '32k ...","{'subreddit': 'wallstreetbets', 'title': 'some...","{'subreddit': 'wallstreetbets', 'title': 'Dail...","{'subreddit': 'wallstreetbets', 'title': 'Casi...","{'subreddit': 'wallstreetbets', 'title': 'Rega...","{'subreddit': 'wallstreetbets', 'title': 'Shop...","{'subreddit': 'wallstreetbets', 'title': 'I th...","{'subreddit': 'wallstreetbets', 'title': 'CVNA..."
3,"{'subreddit': 'wallstreetbets', 'title': 'Week...","{'subreddit': 'wallstreetbets', 'title': 'What...","{'subreddit': 'wallstreetbets', 'title': 'US g...","{'subreddit': 'wallstreetbets', 'title': 'The ...","{'subreddit': 'wallstreetbets', 'title': 'Orac...","{'subreddit': 'wallstreetbets', 'title': 'US w...","{'subreddit': 'wallstreetbets', 'title': 'Why ...","{'subreddit': 'wallstreetbets', 'title': 'US D...","{'subreddit': 'wallstreetbets', 'title': 'Man,...","{'subreddit': 'wallstreetbets', 'title': 'FUCK...",...,"{'subreddit': 'wallstreetbets', 'title': 'Mode...","{'subreddit': 'wallstreetbets', 'title': 'Am I...","{'subreddit': 'wallstreetbets', 'title': '32k ...","{'subreddit': 'wallstreetbets', 'title': 'some...","{'subreddit': 'wallstreetbets', 'title': 'Dail...","{'subreddi

In [22]:
def scrape_reddit(subreddit):
    all_data=[]
    for index,url in enumerate(subreddit):
        headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36"}
        sr_name=url.split("/")[-2]
        print(sr_name)

        try:
            response=requests.get(url,headers=headers,timeout=15)
            response.raise_for_status()

            soup=bs4.BeautifulSoup(response.content,"html.parser")

            subreddit_data={"name":sr_name,"url":url,"title":soup.title.string if soup.title.string else "No Title"}


            topics=[]

            for heading in soup.find_all(['h1','h2','h3','h4']):
                text=heading.get_text(strip=True)

                if text and len(text)>3:
                    if any(keyword in text.lower() for keyword in ['rise','oil','bullish','stocks','bonds','tax','chips','luxury']):
                        topics.append({"title":text,"subreddit":sr_name,"type":"topic"})
            discussions=[]
            seen_url=set()

            for link in soup.find_all('a',href=True):
                text=link.get_text(strip=True)
                href=link['href']
                if text and len(text)>1 and ('/comments/' in href) and (href not in seen_url):
                    seen_url.add(href)
                    discussions.append({"title":text,"url":href,"type":"discussions"})

            subreddit_data["topic"]=topics
            subreddit_data["discussions"]=discussions


            all_data.append(subreddit_data)
            time.sleep(2)

        except Exception as e:
            print(e)

    return all_data

def save_scraped_data(data, filename_json, filename_csv):
    if not data:
        print(f'No data')
        return


    try:
        with open(filename_json, "w", encoding="utf-8") as file:
            json.dump(data, file, indent=2, ensure_ascii=True)
        print(f'Topics saved: {filename_json}')
    except Exception as e:
        print("ERROR:",e)



    try:
        with open(filename_csv, 'w', newline='', encoding='utf-8') as file:
            writer = csv.writer(file)
            writer.writerow(['Subreddit', 'Type', 'Title', 'URL'])

            for subreddit_data in data:
                subreddit = subreddit_data['name']

                for topic in subreddit_data.get('topics', []):
                    writer.writerow([subreddit, topic['type'], topic['title'], ''])

                for discussion in subreddit_data.get('discussions', []):
                    writer.writerow([subreddit, discussion['type'], discussion['title'], discussion['url']])

            print(f'All topics saved to files!')
    except Exception as e:
        print("Error:",e)

In [26]:
a=scrape_reddit(subreddit)
save_scraped_data(a,"smth.json","smth.csv")
a

[{'name': 'wallstreetbets',
  'url': 'https://www.reddit.com/r/wallstreetbets/',
  'title': 'Reddit - The heart of the internet',
  'topic': [],
  'discussions': [{'title': 'Weekly Earnings Threads 3/2 - 3/6votes •comments',
    'url': '/r/wallstreetbets/comments/1rgbpm8/weekly_earnings_threads_32_36/',
    'type': 'discussions'},
   {'title': 'What Are Your Moves Tomorrow, March 06, 2026',
    'url': '/r/wallstreetbets/comments/1rltgpe/what_are_your_moves_tomorrow_march_06_2026/',
    'type': 'discussions'},
   {'title': 'US grants waiver to allow India to buy Russian oil amid Iran war',
    'url': '/r/wallstreetbets/comments/1rm15iz/us_grants_waiver_to_allow_india_to_buy_russian/',
    'type': 'discussions'},
   {'title': 'Supreme Court rules that Trump’s sweeping emergency tariffs are illegal | CNN Politics',
    'url': '/r/wallstreetbets/comments/1r9xu7b/supreme_court_rules_that_trumps_sweeping/',
    'type': 'discussions'},
   {'title': 'The market lately',
    'url': '/r/wallstre

In [27]:
path=(r"C:\Users\Asus\PycharmProjects\test_2\smth.csv")
scraped_df=pd.read_csv(path)
scraped_df

,Subreddit,Type,Title,URL
0,wallstreetbets,discussions,Weekly Earnings Threads 3/2 - 3/6votes •comments,/r/wallstreetbets/comments/1rgbpm8/weekly_earn...
1,wallstreetbets,discussions,"What Are Your Moves Tomorrow, March 06, 2026",/r/wallstreetbets/comments/1rltgpe/what_are_yo...
2,wallstreetbets,discussions,US grants waiver to allow India to buy Russian...,/r/wallstreetbets/comments/1rm15iz/us_grants_w...
3,wallstreetbets,discussions,Supreme Court rules that Trump’s sweeping emer...,/r/wallstreetbets/comments/1r9xu7b/supreme_cou...
4,wallstreetbets,discussions,The market lately,/r/wallstreetbets/comments/1rlz6d2/the_market_...
5,stocks,discussions,Rate My Portfolio - r/Stocks Quarterly Thread ...,/r/stocks/comments/1rhtjtk/rate_my_portfolio_r...
6,stocks,discussions,r/Stocks Daily Discussion & Options Trading Th...,/r/stocks/comments/1rle70j/rstocks_daily_discu...
7,stocks,discussions,Trump on rising gas prices during Iran operati...,/r/stocks/comments/1rlqxo4/trump_on_rising_gas...
8,stocks,discussions,Trump’s Global Tariffs Struck Down by US Supre...,/r/stocks/comments/1r9xpm4/trumps_global_tarif...
9,stocks,discussions,Next stocks to be added to s and p 500 in reba...,/r/stocks/comments/1rlt7xm/next_stocks_to_be_a...


In [31]:
scraped_df.iloc[2,2]

'US grants waiver to allow India to buy Russian oil amid Iran war'

In [49]:
all_data=[]         #main list
headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36"}     #framework

for index,url in enumerate(subreddit):
        sr_name="wallstreetbets"      #names of subreddit
        after=None
        posts=[]
        print(sr_name)
        params = {"limit": 100}
        if after:
                params["after"] = after

        try:
                response=requests.get(f"https://www.reddit.com/r/{sr_name}/.json",headers=headers,timeout=15,params=params)
                res_json=response.json()
                data = res_json["data"]["children"]
                response.raise_for_status()
                for post in data:
                    p = post["data"]

                    comment_url = "https://reddit.com" + p["permalink"] + ".json"

                    comment_response = requests.get(comment_url, headers=headers)
                    comment_json = comment_response.json()
                    comments = []

                    for c in comment_json[1]["data"]["children"]:
                        if c["kind"] == "t1":   # t1 = comment
                            comments.append(c["data"]["body"])

                    posts.append({"subreddit": sr_name,"title": p["title"],"url": "https://reddit.com" + p["permalink"],"score": p["score"],"num_comments": p["num_comments"],"comments": comments})



        except Exception as e:
            print(e)

Expecting value: line 1 column 1 (char 0)


In [4]:
def scrape_reddit(subreddit: list, total_posts: int):
    all_data = []  #main list
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36"}  #framework

    for index, url in enumerate(subreddit):
        sr_name = url.split("/")[-2]  #names of subreddit
        after = None
        posts = []
        print(sr_name)

        while len(posts) < total_posts:

            params = {"limit": total_posts}
            #to check the loop may be taking 50 posts then is checking which is the stuff we want etc
            if after:
                params["after"] = after

            try:
                response = requests.get(f"https://www.reddit.com/r/{sr_name}/.json", headers=headers, timeout=15,
                                        params=params)
                res_json = response.json()
                data = res_json["data"]["children"]
                response.raise_for_status()

                for post in data:
                    p = post["data"]

                    comment_url = "https://reddit.com" + p["permalink"] + ".json"

                    comment_response = requests.get(comment_url, headers=headers)
                    comment_json = comment_response.json()
                    comments = []

                    for c in comment_json[1]["data"]["children"]:
                        time.sleep(0.5)
                        if c["kind"] == "t1":   # t1 = comment
                            comments.append(c["data"]["body"])

                    posts.append({"subreddit": sr_name,"title": p["title"],"url": "https://reddit.com" + p["permalink"],"score": p["score"],"num_comments": p["num_comments"],"comments": comments})


                after = res_json["data"]["after"]

                if after is None:
                    break
                time.sleep(1)

                all_data.extend(posts)


            except Exception as e:
                print(e)

    return all_data


def save_scraped_data(data, filename_json, filename_csv):
    if not data:
        print(f'No data')
        return

    try:
        with open(filename_json, "w", encoding="utf-8") as file:
            json.dump(data, file, indent=2, ensure_ascii=True)
        print(f'Topics saved: {filename_json}')
    except Exception as e:
        print("ERROR:", e)

    try:
        with open(filename_csv, 'w', newline='', encoding='utf-8') as file:
            writer = csv.writer(file)
            writer.writerow(['Subreddit', 'Type', 'Title', 'URL'])

            for subreddit_data in data:
                subreddit = subreddit_data['name']

                for topic in subreddit_data.get('topics', []):
                    writer.writerow([subreddit, topic['type'], topic['title'], ''])

                for discussion in subreddit_data.get('discussions', []):
                    writer.writerow([subreddit, discussion['type'], discussion['title'], discussion['url']])

            print(f'All topics saved to files!')
    except Exception as e:
        print("Error:", e)

In [5]:
a=scrape_reddit(subreddit,5)

stocks


In [6]:
a

[{'subreddit': 'wallstreetbets',
  'title': 'Weekly Earnings Threads 3/2 - 3/6',
  'url': 'https://reddit.com/r/wallstreetbets/comments/1rgbpm8/weekly_earnings_threads_32_36/',
  'score': 190,
  'num_comments': 1241,
  'comments': ['Competing against after hours institution news-based Bots that scan RSS feeds and wire services for specific keywords to trade on breaking news instantly.',
   '🥭 meeting with RTX and Lockheed tomorrow to “discuss weapons production”. ',
   'Petrobras 🤔🤔🤔',
   'Full ported gap puts, mrvl calls today',
   'Why is OMDA not moving on banger earnings?',
   'Fucking fuck MRVL shit. I bought calls past 2 earnings for them and got burned. Thought fuck this shit today and went puts. I’m bad at this',
   'Guidewire and pattern will also fly tomorrow.  they came thru with solid beats',
   '[deleted]',
   'Oh man mrvl had a triple beat and had pretty large beat for guidance. \n\nIts up so far, but it will drop 25% knowing this shithole market.\n\n I hope my $85 calls 

In [8]:
a_df=pd.DataFrame(a)
a_df.iloc[1,5]

['What better way to end the week than getting fucked by theta ',
 'Turns out I might have inadvertently bought a war profiteer stock with my rubber plantation a couple of months ago. XD',
 "Hope y'all bought the dip",
 'Buying puts tmr morning 🦅🦅🦅🦅',
 'I’m going to ban myself from the sub for 12 hours. ✌️',
 'I came here and read looking for the meaning of the world and I have not found it',
 'It’s like they say: &gt;!if you weak spaghetti, you’ll mom’s knees.!&lt;',
 'Europe and Asia are green which means gay bers r fuk for the next 6 months',
 'My favorite part of Friday was how the indices and vix were both red. Totally cool and straight. 😮\u200d💨👌',
 'Beautiful all you can buy setup for Bull. ',
 'Why can’t we ban countries that dump overnight from our stock market? Wouldn’t that make number go up?',
 'Why is everyone shorting war gold?',
 'Everyone expecting sell off into weekend, which means inverse and we bull today. \n\nIf no societal collapse over weekend, we rip to new ATH n

In [9]:
a_df.to_csv(r"C:\Users\Asus\PycharmProjects\test_2\smth.csv")

In [ ]:
#soup.prettify makes html look like normal

#soup.find('<h1,h2,h5,div>') gives that tag line it gives first occurence of h5 soup.find_all('<h5>') gives all occurences
#tags=soup.find_all('<h5>')
#for i in tags:
#   i.text gives text values

#fs=soup.find_all('div',class_='card') {class_ here _ is important as class is a keyword in python}
#for i in fs:
#   i.<'h5','h1'> will give h5 tages or h1 tags
#   i.h5.text gives text
#   i.a